# Pre-Main Gate — Cost Critic warm-up + Actor delay 1/2 + N=16

Main設定を凍結する直前の最終Gateです。比較対象は **A Full Priorのみ**で、次だけを比較します。

- 共通: Residual Graph Actor、Prior interaction P2P、Runtime HOCBF、n=10、episode dual、`d_cost=6`
- 共通schedule: fresh rollout収集 → Cost Critic-only warm-up → online CAL
- 比較因子: `actor_update_period = 1` 対 `2`
- N: 2 / 4 / 8 / 16

同一seed内では、収集済みReplay、warm-up済みCost Critic、Actor、Reward Critic、α、λ、乱数状態を共通checkpointから分岐します。したがってdelay以外の初期差を持ち込みません。

`RUN_SCALE='gate'`は長時間実験です。最初の動作確認だけなら`'smoke'`に変更してください。結果は`runs/pre_main_gate/`以下へ保存され、再実行時はcheckpointから再開します。完了済みwarm-upの中間checkpointは検証後に自動削除し、online完了後はReplayを含まない評価用checkpointだけを保存します。

In [1]:
from pathlib import Path
from copy import deepcopy
from datetime import datetime, timezone
import gc, hashlib, json, math, os, platform, random, shutil, webbrowser
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if not (ROOT / 'uav_safe_marl').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'uav_safe_marl').is_dir():
    raise RuntimeError('Repository rootからNotebookを実行してください。')

# ===== User controls =====
RUN_SCALE = 'gate'              # 'smoke' or 'gate'
RUN_NAME = 'pre_main_gate'
TRAINING_SEEDS = [0, 1]         # Gateは2 seed。採用候補だけ後で3 seed目を追加可能
ACTOR_DELAYS = [1, 2]
AGENT_COUNTS = [2, 4, 8, 16]
FORCE_REBUILD_BANKS = False
FORCE_RECOLLECT_WARMUP = False
FORCE_RETRAIN_ARMS = False
FORCE_REEVALUATE = False
AUTO_OPEN_TENSORBOARD = True
SAFETY_BACKEND = 'serial'       # Notebook/Windowsで確実に動く既定値。検証済み環境のみ'auto'へ変更可

if RUN_SCALE == 'smoke':
    WARMUP_SCENARIOS_PER_N = 2
    ONLINE_SCENARIOS_PER_N = 3
    DEVELOPMENT_SCENARIOS_PER_N = 2
    COST_WARMUP_UPDATES = 100
    ONLINE_ENVIRONMENT_STEPS = 2_000
    CALIBRATION_REPEATS = 1
else:
    WARMUP_SCENARIOS_PER_N = 25
    ONLINE_SCENARIOS_PER_N = 40
    DEVELOPMENT_SCENARIOS_PER_N = 8
    COST_WARMUP_UPDATES = 5_000
    ONLINE_ENVIRONMENT_STEPS = 30_000
    CALIBRATION_REPEATS = 2

WARMUP_SAVE_INTERVAL = 500
D_COST = 6.0
COST_N_STEP = 10
OUTPUT = ROOT / 'runs' / RUN_NAME
BANK_DIR = OUTPUT / 'banks'
OUTPUT.mkdir(parents=True, exist_ok=True)
BANK_DIR.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print({
    'root': str(ROOT), 'output': str(OUTPUT), 'scale': RUN_SCALE,
    'device': DEVICE, 'seeds': TRAINING_SEEDS, 'delays': ACTOR_DELAYS,
    'N': AGENT_COUNTS, 'warmup_updates': COST_WARMUP_UPDATES,
    'online_environment_steps_per_arm': ONLINE_ENVIRONMENT_STEPS,
})

{'root': '/Users/iniad/Documents/#lab/Safe-RL_20260915/Codes', 'output': '/Users/iniad/Documents/#lab/Safe-RL_20260915/Codes/runs/pre_main_gate', 'scale': 'gate', 'device': 'mps', 'seeds': [0, 1], 'delays': [1, 2], 'N': [2, 4, 8, 16], 'warmup_updates': 5000, 'online_environment_steps_per_arm': 30000}


## 1. Hardware・実装監査

In [2]:
from uav_safe_marl import build_env, load_config
from uav_safe_marl.evaluation import (
    calibrate_cost_critic, evaluate_scenario_bank, generate_random_traffic_bank,
    load_scenario_bank, save_evaluation, save_scenario_bank,
)
from uav_safe_marl.runners.trainer import SafeRLTrainer

hardware = {
    'platform': platform.platform(), 'python': platform.python_version(),
    'torch': torch.__version__, 'cuda_available': torch.cuda.is_available(),
    'mps_available': torch.backends.mps.is_available(), 'device': DEVICE,
    'logical_cpu_count': os.cpu_count(),
}
if torch.cuda.is_available():
    hardware['cuda_device'] = torch.cuda.get_device_name(0)
print(hardware)

# actor_update_periodが正式Configとして認識されることを先に監査する。
for delay in ACTOR_DELAYS:
    probe = load_config(ROOT / 'configs' / 'full.yaml', overrides={'training': {'actor_update_period': delay}})
    assert probe.training.actor_update_period == delay
print('actor_update_period config audit: passed')

{'platform': 'macOS-15.6-arm64-arm-64bit-Mach-O', 'python': '3.13.3', 'torch': '2.14.0', 'cuda_available': False, 'mps_available': True, 'device': 'mps', 'logical_cpu_count': 12}
actor_update_period config audit: passed


## 2. 固定Scenario bank生成

In [3]:
def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def interleave(by_n):
    result = []
    length = len(next(iter(by_n.values())))
    for index in range(length):
        for n in AGENT_COUNTS:
            result.append(by_n[n][index])
    return result

def generate_balanced_bank(path: Path, per_n: int, root_seed: int):
    if path.exists() and not FORCE_REBUILD_BANKS:
        scenarios = load_scenario_bank(path)
    else:
        by_n = {}
        for offset, n in enumerate(AGENT_COUNTS):
            by_n[n] = generate_random_traffic_bank(
                n, per_n, root_seed + 1009 * offset,
                base_size=280.0, transverse_size=280.0,
                spatial_scaling_mode='fixed', altitude_range=(-28.0, 28.0),
                minimum_initial_distance=20.0,
                d_safe=10.0, d_warn=30.0, d_eng=50.0,
                los_prediction_threshold=75.0, prediction_horizon=50.0,
                initial_speed_fraction_range=(0.40, 0.65),
                acceleration_range=(2.0, 4.0), velocity_limit_range=(10.0, 14.0),
                require_predicted_conflict=True,
            )
        scenarios = interleave(by_n)
        save_scenario_bank(path, scenarios)
        # 初回生成時も再開時も、同じJSON正規形をTrainerへ渡す。
        scenarios = load_scenario_bank(path)
    counts = {n: sum(int(s['actual_agent_count']) == n for s in scenarios) for n in AGENT_COUNTS}
    if any(counts[n] != per_n for n in AGENT_COUNTS):
        raise RuntimeError(f'Unexpected N distribution in {path}: {counts}')
    if any(len(s['predicted_interaction_edges']) < 1 for s in scenarios):
        raise RuntimeError(f'candidate-zero scenario found in {path}')
    return scenarios

WARMUP_BANK_PATH = BANK_DIR / 'warmup_bank.json'
ONLINE_BANK_PATH = BANK_DIR / 'online_bank.json'
DEVELOPMENT_BANK_PATH = BANK_DIR / 'development_bank.json'
warmup_bank = generate_balanced_bank(WARMUP_BANK_PATH, WARMUP_SCENARIOS_PER_N, 26_091_001)
online_bank = generate_balanced_bank(ONLINE_BANK_PATH, ONLINE_SCENARIOS_PER_N, 26_092_001)
development_bank = generate_balanced_bank(DEVELOPMENT_BANK_PATH, DEVELOPMENT_SCENARIOS_PER_N, 26_093_001)

id_sets = [set(s['scenario_id'] for s in bank) for bank in (warmup_bank, online_bank, development_bank)]
assert not (id_sets[0] & id_sets[1] or id_sets[0] & id_sets[2] or id_sets[1] & id_sets[2])
bank_hashes = {p.stem: sha256(p) for p in (WARMUP_BANK_PATH, ONLINE_BANK_PATH, DEVELOPMENT_BANK_PATH)}
maximum_reference_time = max(
    max(s['reference_arrival_times']) for bank in (warmup_bank, online_bank, development_bank) for s in bank
)
MAX_STEPS = int(math.ceil(2.0 * maximum_reference_time / 0.1))
print({
    'warmup_scenarios': len(warmup_bank), 'online_scenarios': len(online_bank),
    'development_scenarios': len(development_bank), 'max_steps': MAX_STEPS,
    'bank_hashes': bank_hashes,
})

{'warmup_scenarios': 100, 'online_scenarios': 160, 'development_scenarios': 32, 'max_steps': 1894, 'bank_hashes': {'warmup_bank': 'aca4d4f80ee8bbd2ff9d4746c5b3aab2612723823a0caac324320f521f378bd4', 'online_bank': 'aad3112d599fcd7a0749c45a502aa4374937cc2d295d00d3214eeab0d832981c', 'development_bank': '8e15aabac2bdc2090ea4c06be779e260c9f495ded392c5e78a4968a9b9553bfa'}}


## 3. 共通設定とTensorBoard

In [4]:
def config_for(seed: int, actor_delay: int, *, online_budget):
    distribution = {n: 1.0 / len(AGENT_COUNTS) for n in AGENT_COUNTS}
    overrides = {
        'seed': seed, 'device': DEVICE,
        'environment': {
            'agent_count': 2, 'max_agent_count': max(AGENT_COUNTS),
            'max_steps': MAX_STEPS, 'goal_tolerance': 5.0,
        },
        'scenario': {
            'spatial_scaling_mode': 'fixed', 'longitudinal_length': 280.0,
            'transverse_size': 280.0, 'altitude_min': -28.0, 'altitude_max': 28.0,
            'minimum_initial_separation': 20.0, 'interaction_threshold': 75.0,
            'prediction_horizon': 50.0, 'initial_speed_fraction_min': 0.40,
            'initial_speed_fraction_max': 0.65, 'require_predicted_candidate': True,
            'training_agent_count_distribution': distribution,
        },
        'control': {
            'actor_mode': 'residual', 'reference_speed_fraction': 0.75,
            'reference_velocity_gain': 0.8, 'residual_acceleration_fraction': 0.5,
        },
        'communication': {'information_mode': 'prior_share', 'actor_period_k': 5, 'delay_steps': 1},
        'graph': {'actor_aggregation_backend': 'graph', 'critic_aggregation_backend': 'graph'},
        'safety': {
            'enabled': True, 'responsibility_mode': 'capability',
            'conflict_resolver_enabled': True, 'd_safe': 10.0, 'd_warn': 30.0,
            'd_eng': 50.0, 'speed_constraint_mode': 'norm',
        },
        'objective': {
            'distance_cost_enabled': True, 'intervention_cost_enabled': True,
            'w_intervention': 1.0, 'goal_reward': 15.0,
            'absolute_distance_enabled': False, 'residual_penalty_enabled': True,
        },
        'learning': {
            'd_cost': D_COST, 'gamma_cost': 0.99, 'cal_lambda_lr': 0.001,
            'cal_mode': 'pdf_variant', 'constraint_semantics': 'episodic_start_state',
            'cal_variant': 'episodic_dual_replay_gradient', 'cost_target_n_step': COST_N_STEP,
            'batch_size': 256, 'buffer_size': 200_000,
            'initial_state_buffer_size': 10_000, 'cost_ensemble_size': 2, 'beta_ucb': 1.0,
        },
        'training': {
            'total_environment_steps_budget': online_budget,
            'learning_starts': 1, 'updates_per_environment_step': 1,
            'actor_update_period': actor_delay, 'checkpoint_every_episodes': 5,
            'dual_learning_starts': 1, 'dual_update_mode': 'episode',
            'dual_update_interval_episodes': 1, 'dual_batch_size': 256,
        },
        'monitoring': {
            'enabled': True, 'progress_bar': True, 'tensorboard': True,
            'auto_launch_tensorboard': False, 'open_browser': False,
            'tensorboard_host': '127.0.0.1', 'tensorboard_port': 0,
            'step_refresh_interval': 25,
        },
        'performance': {'profile': 'exact', 'safety_backend': SAFETY_BACKEND},
    }
    config = load_config(ROOT / 'configs' / 'full.yaml', overrides=overrides)
    assert config.learning.cost_target_n_step == 10
    assert config.training.dual_update_mode == 'episode'
    assert config.learning.d_cost == 6.0
    return config

tensorboard_url = None
try:
    from tensorboard import program
    server = program.TensorBoard()
    server.configure(argv=[None, '--logdir', str(OUTPUT), '--host', '127.0.0.1', '--port', '0'])
    tensorboard_url = server.launch()
    display(Markdown(f'**TensorBoard:** [{tensorboard_url}]({tensorboard_url})'))
    if AUTO_OPEN_TENSORBOARD:
        webbrowser.open(tensorboard_url, new=2)
except (Exception, SystemExit) as exc:
    print(f'TensorBoardを起動できませんでした（JSONL/tqdmは継続）: {exc}')

**TensorBoard:** [http://127.0.0.1:63037/](http://127.0.0.1:63037/)

## 4. Seed別の共通Cost Critic warm-up

各seedで一度だけ収集・warm-upし、その最終checkpointをdelay=1/2の両armが読み込みます。Actor、Reward Critic、Reward target、α、λがwarm-up中に変化していないことをhashで監査します。

In [5]:
def module_hash(module) -> str:
    digest = hashlib.sha256()
    for name, value in sorted(module.state_dict().items()):
        digest.update(name.encode())
        digest.update(value.detach().cpu().numpy().tobytes())
    return digest.hexdigest()

def frozen_hashes(trainer):
    return {
        'actor': module_hash(trainer.backend.actor),
        'reward_critics': module_hash(trainer.backend.reward_critics),
        'reward_targets': module_hash(trainer.backend.reward_targets),
        'log_alpha': hashlib.sha256(trainer.backend.log_alpha.detach().cpu().numpy().tobytes()).hexdigest(),
        'lambda': float(trainer.backend.dual.value),
    }

def collect_and_warm_seed(seed: int) -> Path:
    root = OUTPUT / 'warmup' / f'seed{seed}'
    root.mkdir(parents=True, exist_ok=True)
    collection_ckpt = root / 'collection_checkpoint.pt'
    progress_ckpt = root / 'warmup_progress.pt'
    final_ckpt = root / 'warmup_final.pt'
    progress_json = root / 'warmup_progress.json'
    audit_json = root / 'warmup_audit.json'
    config = config_for(seed, 1, online_budget=None)

    if final_ckpt.exists() and not FORCE_RECOLLECT_WARMUP:
        if not audit_json.exists():
            raise RuntimeError(f'seed{seed}: warm-up auditが見つかりません')
        audit = json.loads(audit_json.read_text(encoding='utf-8'))
        if audit['bank_hashes']['warmup_bank'] != bank_hashes['warmup_bank']:
            raise RuntimeError(f'seed{seed}: warm-up bank hash mismatch')
        print(f'seed{seed}: completed warm-up checkpointを再利用')
        return final_ckpt

    trainer = SafeRLTrainer(config, build_env(config), root / 'collection')
    if collection_ckpt.exists() and not FORCE_RECOLLECT_WARMUP:
        trainer.load_checkpoint(collection_ckpt, resume_training=True)
        trainer.logger.truncate_after_episode(trainer.completed_episodes)
    remaining = len(warmup_bank) - trainer.completed_episodes
    if remaining > 0:
        trainer.train(
            episodes=remaining, scenarios=warmup_bank, live_display=True,
            checkpoint_path=collection_ckpt, update_mode='collect_only',
        )
        trainer.save_checkpoint(collection_ckpt)
    if trainer.scenario_bank_hash != bank_hashes['warmup_bank']:
        # Trainer hashes the serialized list, while the bank hash is the file hash.
        # The file hash is stored explicitly below; scenario identity is already audited on load.
        pass

    reference = frozen_hashes(trainer)
    completed_updates = 0
    if progress_ckpt.exists() and progress_json.exists() and not FORCE_RECOLLECT_WARMUP:
        trainer.load_checkpoint(progress_ckpt, resume_training=True)
        progress = json.loads(progress_json.read_text(encoding='utf-8'))
        completed_updates = int(progress['completed_updates'])
        reference = progress['frozen_reference']

    update_log = root / 'warmup_updates.jsonl'
    mode = 'a' if completed_updates else 'w'
    with update_log.open(mode, encoding='utf-8') as stream:
        progress = tqdm(range(completed_updates, COST_WARMUP_UPDATES), desc=f'seed{seed} Cost warm-up n=10')
        for index in progress:
            batch = trainer.buffer.sample_n_step(
                config.learning.batch_size, COST_N_STEP, config.learning.gamma_cost
            )
            metrics = trainer.backend.update_cost_critic_only_n_step(batch)
            update_number = index + 1
            if update_number % 100 == 0:
                stream.write(json.dumps({'seed': seed, 'update': update_number, **metrics}) + '\n')
                stream.flush()
                progress.set_postfix(loss=f"{metrics['cost_critic_loss']:.3f}", ucb=f"{metrics['cost_ucb_batch_mean']:.2f}")
            if update_number % WARMUP_SAVE_INTERVAL == 0 or update_number == COST_WARMUP_UPDATES:
                trainer.save_checkpoint(progress_ckpt)
                progress_json.write_text(json.dumps({
                    'completed_updates': update_number,
                    'frozen_reference': reference,
                    'warmup_bank_file_sha256': bank_hashes['warmup_bank'],
                }, indent=2), encoding='utf-8')

    after = frozen_hashes(trainer)
    if after != reference:
        raise RuntimeError(f'seed{seed}: non-Cost parameters changed during warm-up')
    trainer.save_checkpoint(final_ckpt)
    payload = torch.load(final_ckpt, map_location='cpu', weights_only=False)
    payload['experiment_identity']['scenario_bank_sha256'] = bank_hashes['warmup_bank']
    temporary = final_ckpt.with_suffix('.tmp')
    torch.save(payload, temporary)
    os.replace(temporary, final_ckpt)
    audit_json.write_text(json.dumps({
        'seed': seed, 'warmup_scenarios': len(warmup_bank),
        'replay_transitions': len(trainer.buffer),
        'replay_by_n': trainer.buffer.occupancy_by_agent_count(),
        'cost_warmup_updates': COST_WARMUP_UPDATES,
        'cost_n_step': COST_N_STEP, 'frozen_audit': 'passed',
        'frozen_hashes': after, 'bank_hashes': bank_hashes,
    }, indent=2), encoding='utf-8')
    trainer.env.close()
    del trainer
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    if torch.backends.mps.is_available(): torch.mps.empty_cache()
    return final_ckpt

warmup_checkpoints = {seed: collect_and_warm_seed(seed) for seed in TRAINING_SEEDS}
# final checkpointのload/hash監査後は、同じReplayを含む中間snapshotは不要。
reclaimed = 0
for seed, final_path in warmup_checkpoints.items():
    root = final_path.parent
    for name in ('collection_checkpoint.pt', 'warmup_progress.pt'):
        candidate = root / name
        if candidate.exists():
            reclaimed += candidate.stat().st_size
            candidate.unlink()
    for candidate in root.glob('*.tmp'):
        reclaimed += candidate.stat().st_size
        candidate.unlink()
free_gib = shutil.disk_usage(OUTPUT).free / 1024**3
print({seed: str(path) for seed, path in warmup_checkpoints.items()})
print({'reclaimed_intermediate_GiB': reclaimed / 1024**3, 'free_disk_GiB': free_gib})

seed0: completed warm-up checkpointを再利用
seed1: completed warm-up checkpointを再利用
{0: '/Users/iniad/Documents/#lab/Safe-RL_20260915/Codes/runs/pre_main_gate/warmup/seed0/warmup_final.pt', 1: '/Users/iniad/Documents/#lab/Safe-RL_20260915/Codes/runs/pre_main_gate/warmup/seed1/warmup_final.pt'}


## 5. 共通warm-upからdelay=1/2へ分岐してonline CAL

Warm-up収集stepはonline budgetへ含めません。両armは同じwarm-up済み状態から開始し、online部分だけ同じenvironment-step budgetで比較します。

In [6]:
def load_warmup_into_arm(trainer, warmup_path: Path):
    payload = torch.load(warmup_path, map_location=trainer.backend.device, weights_only=False)
    trainer.backend.load_state_dict(payload['backend'])
    trainer.buffer.load_state_dict(payload['replay_buffer'])
    trainer.initial_state_buffer.load_state_dict(payload['initial_state_buffer'])
    # Online比較のstep/episodeカウンタはゼロから。Replayとinitial-state分布はwarm-upから継承。
    trainer.total_environment_steps = 0
    trainer.total_gradient_updates = 0
    trainer.completed_episodes = 0
    trainer.transitions_collected_by_n = {}
    trainer.scenario_bank_cursor = 0
    trainer.scenario_bank_hash = None

def seed_branch_rng(seed: int):
    branch_seed = 800_000 + seed
    random.seed(branch_seed)
    np.random.seed(branch_seed)
    torch.manual_seed(branch_seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(branch_seed)
    if torch.backends.mps.is_available(): torch.mps.manual_seed(branch_seed)

def normalized_scenario_hash(scenarios) -> str:
    serialized = json.dumps(list(scenarios), sort_keys=True, separators=(',', ':'), default=str).encode()
    return hashlib.sha256(serialized).hexdigest()

def verify_and_migrate_legacy_bank_hash(trainer, arm_root: Path) -> None:
    """初回NumPy表現と再開時JSON表現のhash差だけを安全に移行する。"""
    normalized_hash = normalized_scenario_hash(online_bank)
    if trainer.scenario_bank_hash == normalized_hash:
        return
    audit_path = OUTPUT / 'warmup' / f'seed{trainer.config.seed}' / 'warmup_audit.json'
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    if audit['bank_hashes']['online_bank'] != sha256(ONLINE_BANK_PATH):
        raise RuntimeError('online bank file SHA mismatch; legacy hash migrationを拒否')
    metrics_path = arm_root / 'training' / 'metrics.jsonl'
    rows = [json.loads(line) for line in metrics_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    if len(rows) < trainer.completed_episodes:
        raise RuntimeError('resume checkpointよりmetricsが短いためbank順序を監査できません')
    for row in rows[:trainer.completed_episodes]:
        episode = int(row['episode'])
        expected = online_bank[episode % len(online_bank)]['scenario_id']
        if row['scenario_id'] != expected:
            raise RuntimeError(f'scenario sequence mismatch at episode {episode}')
    trainer.scenario_bank_hash = normalized_hash
    print('legacy in-memory bank hashを、監査済みJSON正規hashへ移行')

def save_evaluation_checkpoint(trainer, path: Path) -> None:
    """Replayを除いた、評価とCalibration用の小型checkpoint。"""
    config_payload = trainer.config.to_dict()
    config_hash = hashlib.sha256(
        json.dumps(config_payload, sort_keys=True, separators=(',', ':')).encode()
    ).hexdigest()
    payload = {
        'schema_version': trainer.CHECKPOINT_SCHEMA_VERSION,
        'backend': trainer.backend.state_dict(),
        'config': config_payload,
        'experiment_identity': {
            'config_sha256': config_hash,
            'scenario_bank_sha256': trainer.scenario_bank_hash,
            'training_seed': trainer.config.seed,
            'source_sha256': trainer.source_hash,
            'checkpoint_scope': 'evaluation_without_replay',
        },
        'trainer_progress': {
            'total_environment_steps': trainer.total_environment_steps,
            'total_gradient_updates': trainer.total_gradient_updates,
            'completed_episodes': trainer.completed_episodes,
        },
    }
    temporary = path.with_name(path.name + '.tmp')
    torch.save(payload, temporary)
    os.replace(temporary, path)

def train_arm(seed: int, delay: int) -> Path:
    arm_root = OUTPUT / 'online' / f'delay{delay}' / f'seed{seed}'
    arm_root.mkdir(parents=True, exist_ok=True)
    checkpoint = arm_root / 'checkpoint.pt'
    final_checkpoint = arm_root / 'final.pt'
    temporary_checkpoint = checkpoint.with_name(checkpoint.name + '.tmp')
    if temporary_checkpoint.exists():
        temporary_checkpoint.unlink()
    free_gib = shutil.disk_usage(OUTPUT).free / 1024**3
    if free_gib < 20.0:
        raise RuntimeError(f'checkpoint保存用の空き容量が不足: {free_gib:.1f} GiB (20 GiB以上必要)')
    config = config_for(seed, delay, online_budget=ONLINE_ENVIRONMENT_STEPS)
    trainer = SafeRLTrainer(config, build_env(config), arm_root / 'training')

    if final_checkpoint.exists() and not FORCE_RETRAIN_ARMS:
        print(f'delay={delay}, seed={seed}: completed armを再利用')
        trainer.env.close()
        return final_checkpoint
    if checkpoint.exists() and not FORCE_RETRAIN_ARMS:
        trainer.load_checkpoint(checkpoint, resume_training=True)
        trainer.logger.truncate_after_episode(trainer.completed_episodes)
        verify_and_migrate_legacy_bank_hash(trainer, arm_root)
        print(f'delay={delay}, seed={seed}: env_step={trainer.total_environment_steps}から再開')
    else:
        load_warmup_into_arm(trainer, warmup_checkpoints[seed])
        seed_branch_rng(seed)  # delay arm間で同じ開始乱数

    if trainer.total_environment_steps < ONLINE_ENVIRONMENT_STEPS:
        trainer.train(
            episodes=None, scenarios=online_bank, live_display=True,
            checkpoint_path=checkpoint, update_mode='full',
        )
    save_evaluation_checkpoint(trainer, final_checkpoint)
    checkpoint.unlink(missing_ok=True)
    trainer.env.close()
    del trainer
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    if torch.backends.mps.is_available(): torch.mps.empty_cache()
    return final_checkpoint

arm_checkpoints = {}
for seed in TRAINING_SEEDS:
    for delay in ACTOR_DELAYS:
        arm_checkpoints[(delay, seed)] = train_arm(seed, delay)
print({str(key): str(value) for key, value in arm_checkpoints.items()})

delay=1, seed=0: env_step=14097から再開


Training: 0episode [00:00, ?episode/s]

ValueError: scenario bank differs from the resumed checkpoint

## 6. Development bank評価

- Primary/Gate性能：deterministic mean-action rollout
- Cost Critic Calibration：SAC stochastic continuationと同じpolicy semantics

両者を混同せず別ファイルへ保存します。実測discounted Cost平均とbudget gapを第6評価指標として追加します。

In [ ]:
def aggregate_realized_cost(rows):
    values = np.asarray([float(r['discounted_episode_cost']) for r in rows], dtype=float)
    result = {
        'realized_discounted_cost': float(values.mean()),
        'realized_discounted_cost_median': float(np.median(values)),
        'budget_gap': float(values.mean() - D_COST),
        'episode_budget_exceedance_rate_diagnostic': float(np.mean(values > D_COST)),
    }
    result['by_n'] = {}
    for n in AGENT_COUNTS:
        group = np.asarray([float(r['discounted_episode_cost']) for r in rows if int(r['actual_agent_count']) == n])
        result['by_n'][str(n)] = {
            'count': int(len(group)), 'mean': float(group.mean()),
            'median': float(np.median(group)), 'budget_gap': float(group.mean() - D_COST),
            'episode_budget_exceedance_rate_diagnostic': float(np.mean(group > D_COST)),
        }
    return result

def load_jsonl(path: Path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def evaluate_arm(seed: int, delay: int):
    arm_root = OUTPUT / 'online' / f'delay{delay}' / f'seed{seed}'
    report_path = arm_root / 'gate_report.json'
    if report_path.exists() and not FORCE_REEVALUATE:
        return json.loads(report_path.read_text(encoding='utf-8'))
    config = config_for(seed, delay, online_budget=ONLINE_ENVIRONMENT_STEPS)
    trainer = SafeRLTrainer(config, build_env(config), arm_root / 'evaluation_runtime')
    trainer.load_checkpoint(arm_checkpoints[(delay, seed)], resume_training=False)
    try:
        deployment = evaluate_scenario_bank(
            trainer, development_bank, method=f'A_Full_Prior_delay{delay}', training_seed=seed
        )
        save_evaluation(deployment, arm_root / 'development_evaluation')
        calibration = calibrate_cost_critic(
            trainer, development_bank, repeats=CALIBRATION_REPEATS,
            output_dir=arm_root / 'cost_calibration',
        )
    finally:
        trainer.env.close()
    rows = deployment['episode_metrics']
    realized = aggregate_realized_cost(rows)
    training_rows = load_jsonl(arm_root / 'training' / 'metrics.jsonl')
    finite_keys = ('actor_loss', 'reward_critic_loss', 'cost_critic_loss', 'lambda', 'discounted_episode_cost')
    numerical_finite = all(
        np.isfinite(float(row[key])) for row in training_rows for key in finite_keys
        if key in row and row[key] is not None
    )
    report = {
        'seed': seed, 'actor_update_period': delay,
        'design': {
            'warmup_scenarios_per_n': WARMUP_SCENARIOS_PER_N,
            'cost_warmup_updates': COST_WARMUP_UPDATES, 'cost_n_step': COST_N_STEP,
            'online_environment_steps_budget': ONLINE_ENVIRONMENT_STEPS,
            'd_cost': D_COST, 'dual_update_mode': 'episode', 'N': AGENT_COUNTS,
            'bank_hashes': bank_hashes,
        },
        'training': {
            'completed_episodes': len(training_rows),
            'final_environment_steps': int(training_rows[-1]['cumulative_environment_steps']),
            'final_lambda': float(training_rows[-1]['lambda']),
            'max_lambda': float(max(r['lambda'] for r in training_rows)),
            'max_abs_actor_loss': float(max(abs(r.get('actor_loss', 0.0)) for r in training_rows)),
            'numerical_finite': bool(numerical_finite),
        },
        'primary_metrics': deployment['primary_metrics'],
        'primary_metrics_by_n': deployment['primary_metrics_by_n'],
        'realized_cost': realized,
        'cost_calibration': {k: v for k, v in calibration.items() if k != 'rows'},
    }
    report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    return report

arm_reports = {
    (delay, seed): evaluate_arm(seed, delay)
    for delay in ACTOR_DELAYS for seed in TRAINING_SEEDS
}
print('development evaluation complete')

## 7. Seed集約・N=16 Gate・delay比較

In [ ]:
def mean_metric(values):
    finite = [float(v) for v in values if v is not None and np.isfinite(float(v))]
    return None if not finite else float(np.mean(finite))

aggregate = {}
for delay in ACTOR_DELAYS:
    reports = [arm_reports[(delay, seed)] for seed in TRAINING_SEEDS]
    aggregate[str(delay)] = {
        'seed_count': len(reports),
        'success': mean_metric([r['primary_metrics']['episode_success_rate'] for r in reports]),
        'separation_violation': mean_metric([r['primary_metrics']['separation_violation_rate'] for r in reports]),
        'path_stretch': mean_metric([r['primary_metrics']['path_stretch'] for r in reports]),
        'pure_intervention': mean_metric([r['primary_metrics']['pure_safety_intervention_magnitude'] for r in reports]),
        'communication_load': mean_metric([r['primary_metrics']['communication_load'] for r in reports]),
        'realized_discounted_cost': mean_metric([r['realized_cost']['realized_discounted_cost'] for r in reports]),
        'budget_gap': mean_metric([r['realized_cost']['budget_gap'] for r in reports]),
        'final_lambda': mean_metric([r['training']['final_lambda'] for r in reports]),
        'max_lambda': mean_metric([r['training']['max_lambda'] for r in reports]),
        'calibration_mae': mean_metric([r['cost_calibration']['overall']['ucb_estimate_mae'] for r in reports]),
        'calibration_correlation': mean_metric([r['cost_calibration']['overall']['correlation'] for r in reports]),
        'calibration_coverage': mean_metric([r['cost_calibration']['overall']['empirical_ucb_coverage'] for r in reports]),
        'n16': {
            'success': mean_metric([r['primary_metrics_by_n']['16']['episode_success_rate'] for r in reports]),
            'separation_violation': mean_metric([r['primary_metrics_by_n']['16']['separation_violation_rate'] for r in reports]),
            'path_stretch': mean_metric([r['primary_metrics_by_n']['16']['path_stretch'] for r in reports]),
            'pure_intervention': mean_metric([r['primary_metrics_by_n']['16']['pure_safety_intervention_magnitude'] for r in reports]),
            'realized_discounted_cost': mean_metric([r['realized_cost']['by_n']['16']['mean'] for r in reports]),
            'budget_gap': mean_metric([r['realized_cost']['by_n']['16']['budget_gap'] for r in reports]),
            'calibration_mae': mean_metric([r['cost_calibration']['by_agent_count']['16']['ucb_estimate_mae'] for r in reports]),
        },
        'all_numerically_finite': all(r['training']['numerical_finite'] for r in reports),
    }

# 自動判定は参考値。最終採用は表・曲線・seed差を見て決める。
eligible = [d for d in ACTOR_DELAYS if aggregate[str(d)]['all_numerically_finite']]
if not eligible:
    provisional = None
elif len(eligible) == 1:
    provisional = eligible[0]
else:
    d1, d2 = eligible
    c1, c2 = aggregate[str(d1)]['realized_discounted_cost'], aggregate[str(d2)]['realized_discounted_cost']
    if c2 < 0.95 * c1:
        provisional = d2
    elif c1 < 0.95 * c2:
        provisional = d1
    else:
        s1, s2 = aggregate[str(d1)]['success'], aggregate[str(d2)]['success']
        provisional = d2 if s2 > s1 + 0.02 else d1

gate_report = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'definition': 'Pre-Main Gate: shared Cost warm-up, actor delay 1 vs 2, N=2/4/8/16',
    'aggregate': aggregate,
    'provisional_recommended_actor_update_period': provisional,
    'recommendation_is_automatic_screening_only': True,
    'required_manual_checks': [
        'Cost Critic follows the changing Actor', 'Actor loss and lambda do not diverge',
        'MC Cost and Pure Intervention trend downward', 'Success/Path Stretch remain acceptable',
        'N=16 runtime and Cost scale remain viable',
    ],
    'arm_reports': {f'delay{d}_seed{s}': r for (d, s), r in arm_reports.items()},
}
(OUTPUT / 'final_gate_report.json').write_text(
    json.dumps(gate_report, ensure_ascii=False, indent=2), encoding='utf-8'
)
display(Markdown('### Aggregate'))
display(aggregate)
print({'provisional_recommended_actor_update_period': provisional,
       'report': str(OUTPUT / 'final_gate_report.json')})

## 8. 学習曲線

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8), constrained_layout=True)
colors = {1: 'tab:blue', 2: 'tab:orange'}
for delay in ACTOR_DELAYS:
    for seed in TRAINING_SEEDS:
        path = OUTPUT / 'online' / f'delay{delay}' / f'seed{seed}' / 'training' / 'metrics.jsonl'
        rows = load_jsonl(path)
        x = np.asarray([r['cumulative_environment_steps'] for r in rows])
        label = f'delay={delay}, seed={seed}'
        style = '-' if seed == TRAINING_SEEDS[0] else '--'
        axes[0,0].plot(x, [r['discounted_episode_cost'] for r in rows], style, alpha=.75, color=colors[delay], label=label)
        axes[0,1].plot(x, [r['lambda'] for r in rows], style, alpha=.75, color=colors[delay])
        axes[0,2].plot(x, [r.get('start_state_cost_ucb', np.nan) for r in rows], style, alpha=.75, color=colors[delay])
        axes[1,0].plot(x, [r.get('actor_loss', np.nan) for r in rows], style, alpha=.75, color=colors[delay])
        axes[1,1].plot(x, [r['pure_safety_intervention_magnitude'] for r in rows], style, alpha=.75, color=colors[delay])
        axes[1,2].plot(x, [r['episode_success'] for r in rows], style, alpha=.75, color=colors[delay])
axes[0,0].axhline(D_COST, color='black', linestyle=':', label='d_cost')
titles = ['Discounted episode Cost', 'Lambda', 'Start-state Cost-UCB', 'Actor loss', 'Pure Intervention', 'Episode success']
for ax, title in zip(axes.flat, titles):
    ax.set_title(title); ax.set_xlabel('environment steps'); ax.grid(alpha=.25)
axes[0,0].legend(fontsize=8)
figure_path = OUTPUT / 'pre_main_gate_curves.png'
fig.savefig(figure_path, dpi=170)
plt.show()
print(figure_path)

## 9. Gate後の判断

次の条件を満たす場合だけ設定を凍結します。

1. 全seed・N=16で数値発散、NaN、Replay/QP異常がない。
2. 実測discounted CostとStart-state UCBの違反方向が概ね整合する。
3. MC CostとPure Interventionが低下し、SuccessとPath Stretchを破壊しない。
4. delay=2が明確に安定なら2、差が小さければ単純なdelay=1を採用する。
5. `d_cost=6`はN別MC Costで経験的達成可能性を判断し、Critic誤差だけを理由に変更しない。

このNotebookで使用したDevelopment bankは設定選択用です。Main設定凍結後は、Training / Development / 完全未使用Test bankを新規生成してください。